### Testing generation and comparing results for basic LLM, standard RAG, and this NIR pipeline

This notebook evaluates the following pipelines to determine the most effective approach for generating answers:
1. Basic LLM: uses a general world overview description as context and generates a narrative element based on the query and this text
2. Standard RAG: creates a vector database with text fragments and generates a narrative element based on the query and retrieved context
3. This NIR pipeline (version with a pre-stage plan generation): retrieves context from a graph and then, first, generates an answer plan based on the query and context; second, generates the final answer using the plan and context (based on the query)
4. This NIR pipeline (version without pre-stages): retrieves context from a graph and then generates an answer based on the query and context

**Metrics used:**

RAG efficiency and world consistency:
1. Faithfulness (RAGAS) – measures how factually consistent the generated answer is with the provided context. It evaluates whether the answer is fully grounded in the retrieved information and does not introduce unsupported or hallucinated statements. It is computed by comparing each claim in the generated answer against the retrieved context and checking whether it can be directly inferred from it. Higher values indicate that the model strictly follows the given context without adding external or fabricated information.
2. Answer Relevancy (RAGAS) – measures how relevant the generated answer is to the input question. It evaluates whether the response directly addresses the query without drifting into unrelated information. It is typically computed by comparing semantic similarity between the question and the generated answer. Higher scores indicate that the answer is well-aligned with the user’s intent.
3. Context Precision (RAGAS) – measures how relevant the retrieved context passages are to the question. It evaluates the proportion of useful retrieved information compared to all retrieved context. It is computed by checking which retrieved chunks are actually relevant for answering the query. Higher values indicate that the retrieval step returns mostly useful and non-noisy information.
4. Context Recall (RAGAS) – measures how well the retrieved context covers all the information needed to answer the question. It evaluates whether all necessary supporting facts are present in the retrieved context. It is computed by comparing the required information for a correct answer with the retrieved passages. Higher values indicate that the retrieval system successfully captures most or all relevant knowledge.
5. BERTScore (Generated Text vs World Description) – measures semantic similarity between the generated text and the world description. It evaluates how well the generated content aligns with the predefined world context in terms of meaning rather than exact wording. The metric is computed using contextual embeddings by matching tokens between the generated text and the world description and calculating precision, recall, and F1 over these matches. Higher values indicate stronger consistency of the generated output with the established world setting.
6. BERTScore (Generated Text vs Ground Truth) – measures semantic similarity between the generated text and the reference (ground truth) answer. It evaluates how closely the model’s output matches the expected correct response in meaning. The score is computed using contextual embeddings in the same way as above, by aligning tokens between generated and reference texts. Higher values indicate that the generated answer is closer in meaning to the ground truth, even if the wording differs.
7. World Consistency (LLM-based evaluation) – evaluates whether the generated text is consistent with the given world description using LLM as a judge. The model is prompted to assess if the generated narrative element could exist within the defined world rules, lore, and constraints (information based on world description), and outputs a continuous score between 0 and 1. Higher scores indicate that the generated text is coherent with the world setting and does not violate its established rules or context.

Text and generated narrative element quality:
1. Distinct-2 – measures lexical diversity of the generated text by computing the ratio of unique bigrams (2-grams) to the total number of bigrams. It evaluates how varied the text is in terms of local word sequences. Higher values indicate more diverse and less repetitive language, while lower values suggest repetitive or templated phrasing.
2. Repetition-2 – measures the level of repetition in the generated text by calculating how often bigrams (2-grams) are repeated within the output. It captures redundancy and looping patterns in phrasing. Lower values indicate more fluent and non-redundant text, while higher values suggest repetitive or overly formulaic generation.
3. MAUVE – measures the distributional similarity between the generated text and reference human-written text distributions. It evaluates how close the model’s output is to human-like text in a probabilistic embedding space. The metric compares clusters of token embeddings from both distributions and quantifies their divergence. Higher MAUVE scores indicate that generated text is more similar to human-written text in style and structure.
4. Self-BLEU – measures diversity within a set of generated texts by treating each generated sample as a hypothesis and the rest as references. It computes BLEU scores across generated outputs to evaluate how similar they are to each other. Lower values indicate higher diversity among generated samples, while higher values suggest that outputs are too similar or repetitive across generations.
5. Interestingness (LLM-as-a-judge) – evaluates how interesting, engaging, and creatively meaningful the generated text is using an LLM as a judge. The metric outputs a score from 0 to 1. It considers several factors: (1) whether the content appropriately reflects choice and agency when the task allows it, (2) how diverse and stylistically appropriate the text is, including whether it fits the tone, style, and world of the game without being overly dramatic or inconsistent, and (3) how creative and novel the idea is, including whether it provides fresh insights, new information about the world, or unique player experiences. Higher scores indicate more engaging, original, and well-aligned narrative content.

In [1]:
#imports

import os
import sys
import tqdm
import pandas as pd
import numpy as np
import logging
import warnings
import json
from typing import Any, Dict, List, Optional, Literal
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_community.vectorstores import FAISS
from pandas import json_normalize
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

#some important stuff setup

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(project_root)
sys.path.insert(0, project_root)

results_dir = os.path.join(project_root, "assets", "outputs", "test_results")

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("faiss").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

#this nir imports

from nir.llm.manager import ModelManager
from nir.llm.providers import ModelConfig

from nir.tests.test_datasets import TEST_DATA_LORE_DESCRIPTION, TEST_DATA_DESING_DOCUMENT, TEST_DATA_SCENARIO, TEST_DATA_RUSSIAN_SCENARIO
from nir.tests.evaluator import analyze_generation, compare_pipelines_directional, compute_effect_size
from nir.tests.metrics import compute_mauve_en, compute_mauve_ru, compute_self_bleu, evaluate_ragas_metrics_batch

from nir.graph.graph_storages.networkx_graph import NetworkXGraph
from nir.core.context_retriever import form_context_with_llm, form_context_without_llm
from nir.core.answers_generator import generate_plan
from nir.core.answers_generator import generate_answer_based_on_plan
from nir.core.answers_generator import generate_answer_based_on_context

In [7]:
#models setup

manager = ModelManager()

instruct_model_config = ModelConfig(model_name="llama3.1:8b-instruct-q6_K", temperature=0.0)
instruct_llm = manager.create_chat_model(name="evaluation_model_tests", option="ollama", config=instruct_model_config)

answer_model_config = ModelConfig(model_name="phi4-mini:latest", temperature=0.7)
answer_llm = manager.create_chat_model(name="generation_model_tests", option="ollama", config=answer_model_config)

embeddings_model = manager.create_embedding_model(name="embeddings_tests", option="ollama", model_name="evilfreelancer/enbeddrus:v0.2")

In [8]:
#data setup

test_data_lore_description = TEST_DATA_LORE_DESCRIPTION
test_data_design_document = TEST_DATA_DESING_DOCUMENT
test_data_scenario = TEST_DATA_SCENARIO

In [6]:
#russian tests setup

answer_model_config = ModelConfig(model_name="phi4-mini:latest", temperature=0.7)
answer_llm = manager.create_chat_model(name="russian_generation_model_tests", option="ollama", config=answer_model_config)

instruct_model_config = ModelConfig(model_name="mistral:7b-instruct-q2_K", temperature=0.0)
instruct_llm = manager.create_chat_model(name="evaluation_model_tests", option="ollama", config=instruct_model_config)

test_data_russian = TEST_DATA_RUSSIAN_SCENARIO

**Testing basic LLM**

In [4]:
def run_generation_tests_basic_llm(test_data: Dict[str, Any], dataset_name: str, output_filename: str, language: str="en") -> list[str]:
    all_metrics = []
    generated = []
    references = []

    questions, answers, contexts_raw, ground_truths, categories = [], [], [], [], []

    for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing basic llm generation on {dataset_name}"):
        query = task["query"]
        reference = task["reference"]
        category = task.get("category", "default")
        context = test_data.get("text_summary", "")
        if language == "ru":
            prompt = f"""
                Используй приведенный контекст и напиши ответ за запрос пользователя. 
                Следуй его инструкциям и напиши то, что пользователь от тебя ждет. \nКонтекст:\n{context}\nЗапрос\n{query}"
            """
        else:
            prompt = f"""
                Use provided context to answer user's query.
                Follow their insruction and write that user is expecting from you. \nContext:\n{context}\nQuery:\n{query}
            """
        answer_final = answer_llm.invoke(prompt)

        generated.append({"category": category, "generated_text": answer_final})
        references.append(reference)
        questions.append(query)
        answers.append(answer_final)
        contexts_raw.append(context)
        ground_truths.append(reference)
        categories.append(category)

        metrics = analyze_generation(
            generated_text=answer_final,
            context=context,
            lore_summary=context,
            reference_text=reference,
            query=query,
            category=category,
            evaluation_llm=instruct_llm,
            language="en"
        )

        all_metrics.append(metrics)

    generated_texts = answers.copy()
    # print("Running batch RAGAS evaluation...")
    # ragas_results = evaluate_ragas_metrics_batch(
    #     questions=questions,
    #     answers=answers,
    #     contexts_raw=contexts_raw,
    #     ground_truths=ground_truths,
    # )
    # for i in range(len(all_metrics)):
    #     all_metrics[i].update(ragas_results[i])
        
    if language == "ru":
        mauve_score = compute_mauve_ru(generateds=generated_texts, references=references)
    else:
        mauve_score = compute_mauve_en(generateds=generated_texts, references=references)
    self_bleu_score = compute_self_bleu(generateds=generated_texts)

    print(f"RESULT FOR {dataset_name.capitalize()}")
    if not all_metrics:
        display(pd.DataFrame({"status": ["No data for analysis"]}))
    else:
        df = pd.DataFrame(all_metrics)
        num_cols = df.select_dtypes(include="number").columns.tolist()

        if "category" in df.columns and len(df) > 0:
            cat_df = df.groupby("category")[num_cols].mean().reset_index()
        else:
            cat_df = df[num_cols].mean().to_frame().T
            cat_df["category"] = "default"

        overall = {col: df[col].mean() for col in num_cols}
        overall["category"] = "OVERALL"
        overall_df = pd.DataFrame([overall])

        final_df = pd.concat([cat_df, overall_df], ignore_index=True)
        cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
        final_df = final_df[cols_order]
        
        def pretty_table(df):
            display(HTML("""
                <style>
                table {
                    width: 100%;
                    table-layout: fixed;
                    max-width: 1500px;
                }
                th {
                    word-break: break-word;
                    white-space: normal;
                }
                td {
                    word-break: break-word;
                    white-space: normal;
                }
                </style>
            """))
            display(df.style.hide(axis="index"))
        pretty_table(final_df)

    print(f"Mauve metric for texts generated on {dataset_name}: {mauve_score}")
    print(f"Self-BLEU metric for texts generated on {dataset_name}: {self_bleu_score}")

    if generated:
        pipeline_dir = os.path.join(results_dir, "Basic LLM")
        os.makedirs(pipeline_dir, exist_ok=True)

        generated_with_metrics = []
        for i, gen_item in enumerate(generated):
            metrics_item = all_metrics[i] if i < len(all_metrics) else {}
            combined_entry = {
                "generated_text": gen_item["generated_text"],
                "category": gen_item["category"],
                "reference_text": references[i] if i < len(references) else None,
                "metrics": metrics_item
            }
            generated_with_metrics.append(combined_entry)

        output_json_path = os.path.join(pipeline_dir, output_filename)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(generated_with_metrics, f, ensure_ascii=False, indent=2)

        print(f"Generated texts are saved here: {output_json_path}")

    return generated_texts

In [ ]:
texts_lore = run_generation_tests_basic_llm(test_data_lore_description, "lore description", "lore_description.json")
texts_design = run_generation_tests_basic_llm(test_data_design_document, "design document", "design_document.json")
texts_scenario = run_generation_tests_basic_llm(test_data_scenario, "scenario", "scenario.json")

all_generated_texts = texts_lore + texts_design + texts_scenario
self_bleu_all = compute_self_bleu(generateds=all_generated_texts)
print(f"Self-BLEU on ALL generated texts in english: {self_bleu_all}")

Testing basic llm generation on lore description:  28%|██▊       | 7/25 [44:26<1:38:47, 329.32s/it]

In [ ]:
texts_russian = run_generation_tests_basic_llm(test_data_russian, "scenario in russian", "scenario_russian.json", "ru")

**Testing standard RAG**

In [8]:
def run_generation_tests_standart_rag(test_data: Dict[str, Any], dataset_name: str, output_filename: str, language: str="en") -> list[str]:
    all_metrics = []
    generated = []
    references = []

    questions, answers, contexts_raw, ground_truths, categories = [], [], [], [], []
    
    filepath = test_data["path_to_text"]
    loader = TextLoader(filepath, encoding="utf-8")
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    chunks: List[Document] = text_splitter.split_documents(documents)
    vectorstore = FAISS.from_documents(chunks, embeddings_model)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing standard RAG generation on {dataset_name}"):
        query = task["query"]
        reference = task["reference"]
        category = task.get("category", "default")

        docs = retriever.invoke(query)
        context = "\n\n".join(doc.page_content for doc in docs)
        
        if language == "ru":
            prompt = f"""
                Используй приведенный контекст и напиши ответ за запрос пользователя. 
                Следуй его инструкциям и напиши то, что пользователь от тебя ждет. \nКонтекст:\n{context}\nЗапрос\n{query}"
            """
        else:
            prompt = f"""
                Use provided context to answer user's query.
                Follow their insruction and write that user is expecting from you. \nContext:\n{context}\nQuery:\n{query}
            """
        answer_final = answer_llm.invoke(prompt)

        generated.append({"category": category, "generated_text": answer_final})
        references.append(reference)
        questions.append(query)
        answers.append(answer_final)
        contexts_raw.append(context)
        ground_truths.append(reference)
        categories.append(category)

        metrics = analyze_generation(
            generated_text=answer_final,
            context=context,
            lore_summary=context,
            reference_text=reference,
            query=query,
            category=category,
            evaluation_llm=instruct_llm,
            language="en"
        )

        all_metrics.append(metrics)

    generated_texts = answers.copy()
    # print("Running batch RAGAS evaluation...")
    # ragas_results = evaluate_ragas_metrics_batch(
    #     questions=questions,
    #     answers=answers,
    #     contexts_raw=contexts_raw,
    #     ground_truths=ground_truths,
    # )
    # for i in range(len(all_metrics)):
    #     all_metrics[i].update(ragas_results[i])

    if language == "ru":
        mauve_score = compute_mauve_ru(generateds=generated_texts, references=references)
    else:
        mauve_score = compute_mauve_en(generateds=generated_texts, references=references)
    self_bleu_score = compute_self_bleu(generateds=generated_texts)

    print(f"RESULT FOR {dataset_name.capitalize()}")
    if not all_metrics:
        display(pd.DataFrame({"status": ["No data for analysis"]}))
    else:
        df = pd.DataFrame(all_metrics)
        num_cols = df.select_dtypes(include="number").columns.tolist()

        if "category" in df.columns and len(df) > 0:
            cat_df = df.groupby("category")[num_cols].mean().reset_index()
        else:
            cat_df = df[num_cols].mean().to_frame().T
            cat_df["category"] = "default"

        overall = {col: df[col].mean() for col in num_cols}
        overall["category"] = "OVERALL"
        overall_df = pd.DataFrame([overall])

        final_df = pd.concat([cat_df, overall_df], ignore_index=True)
        cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
        final_df = final_df[cols_order]

        display(final_df.style.format(precision=4))

    print(f"Mauve metric for texts generated on {dataset_name}: {mauve_score}")
    print(f"Self-BLEU metric for texts generated on {dataset_name}: {self_bleu_score}")

    if generated:
        pipeline_dir = os.path.join(results_dir, "Standard RAG")
        os.makedirs(pipeline_dir, exist_ok=True)

        generated_with_metrics = []
        for i, gen_item in enumerate(generated):
            metrics_item = all_metrics[i] if i < len(all_metrics) else {}
            combined_entry = {
                "generated_text": gen_item["generated_text"],
                "category": gen_item["category"],
                "reference_text": references[i] if i < len(references) else None,
                "metrics": metrics_item
            }
            generated_with_metrics.append(combined_entry)

        output_json_path = os.path.join(pipeline_dir, output_filename)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(generated_with_metrics, f, ensure_ascii=False, indent=2)

        print(f"Generated texts are saved here: {output_json_path}")

    return generated_texts

In [9]:
texts_lore = run_generation_tests_standart_rag(test_data_lore_description, "lore description", "lore_description.json")
texts_design = run_generation_tests_standart_rag(test_data_design_document, "design document", "design_document.json")
texts_scenario = run_generation_tests_standart_rag(test_data_scenario, "scenario", "scenario.json")

all_generated_texts = texts_lore + texts_design + texts_scenario
self_bleu_all = compute_self_bleu(generateds=all_generated_texts)
print(f"Self-BLEU on ALL generated texts in english: {self_bleu_all}")

Testing standard RAG generation on lore description: 100%|██████████| 25/25 [1:41:56<00:00, 244.65s/it]


Featurizing p:   0%|          | 0/25 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/25 [00:00<?, ?it/s]

RESULT FOR Lore description


,category,bert_score_reference,bert_score_source,distinct_2,interestingness,repetition_2,world_consistency
0,character descripion,0.8008,0.8052,0.9432,0.8000,0.0405,0.4570
1,character description,0.8113,0.8080,0.9117,0.7375,0.0699,0.9017
2,dialogue,0.8031,0.8175,0.9587,0.6600,0.0342,0.8914
3,item description,0.8059,0.8106,0.9687,0.6800,0.0271,0.9342
4,location description,0.8153,0.8077,0.9147,0.7300,0.0610,0.8942
5,quest,0.7831,0.7961,0.8256,0.5800,0.1018,0.8956
6,OVERALL,0.8033,0.8078,0.9171,0.6800,0.0576,0.8856


...

Featurizing p:   0%|          | 0/25 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/25 [00:00<?, ?it/s]

RESULT FOR Scenario


,category,bert_score_reference,bert_score_source,distinct_2,interestingness,repetition_2,world_consistency
0,character description,0.8157,0.8072,0.9663,0.6600,0.0249,0.7942
1,dialogue,0.7911,0.8233,0.9236,0.6100,0.0553,0.8528
2,item description,0.8408,0.8139,0.9628,0.7000,0.0372,0.9128
3,location description,0.8192,0.8145,0.9350,0.7400,0.0447,0.9342
4,quest,0.8063,0.8152,0.8805,0.5000,0.0772,0.7142
5,OVERALL,0.8146,0.8148,0.9336,0.6420,0.0479,0.8416


Mauve metric for texts generated on scenario: 0.22534903230402542
Self-BLEU metric for texts generated on scenario: 0.07994763227353763
Generated texts are saved here: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\Standard RAG\scenario.json
Self-BLEU on ALL generated texts in english: 0.11816776600640301


In [ ]:
texts_russian = run_generation_tests_standart_rag(test_data_russian, "scenario in russian", "scenario_russian.json", "ru")

**Testing ThisNIRPipeline with two-staged generation**

In [6]:
def run_generation_tests_ThisNIRPipeline_two_stages(test_data: Dict[str, Any], dataset_name: str, output_filename: str, language: str="en") -> list[str]:
    all_metrics = []
    generated = []
    references = []

    questions, answers, contexts_raw, ground_truths, categories = [], [], [], [], []
    
    filepath = test_data["path_to_graph"]
    graph = NetworkXGraph()
    graph.load(filepath)

    for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing This NIR two-staged generation on {dataset_name}"):
        query = task["query"]
        reference = task["reference"]
        category = task.get("category", "default")

        this_graph_embeddings = manager.get_embedding_model(graph.get_embedding_model())

        context = form_context_without_llm(query, graph, this_graph_embeddings, language)
        answer_plan = generate_plan(query, context, answer_llm, False, language)
        answer_final = generate_answer_based_on_plan(query, answer_plan, context, answer_llm, language)

        generated.append({"category": category, "generated_text": answer_final})
        references.append(reference)
        questions.append(query)
        answers.append(answer_final)
        contexts_raw.append(context)
        ground_truths.append(reference)
        categories.append(category)
        
        metrics = analyze_generation(
            generated_text=answer_final,
            context=context,
            lore_summary=context,
            reference_text=reference,
            query=query,
            category=category,
            evaluation_llm=instruct_llm,
            language="en"
        )

        all_metrics.append(metrics)

    generated_texts = answers.copy()
    # print("Running batch RAGAS evaluation...")
    # ragas_results = evaluate_ragas_metrics_batch(
    #     questions=questions,
    #     answers=answers,
    #     contexts_raw=contexts_raw,
    #     ground_truths=ground_truths,
    # )
    # for i in range(len(all_metrics)):
    #     all_metrics[i].update(ragas_results[i])

    if language == "ru":
        mauve_score = compute_mauve_ru(generateds=generated_texts, references=references)
    else:
        mauve_score = compute_mauve_en(generateds=generated_texts, references=references)
    self_bleu_score = compute_self_bleu(generateds=generated_texts)

    print(f"RESULT FOR {dataset_name.capitalize()}")
    if not all_metrics:
        display(pd.DataFrame({"status": ["No data for analysis"]}))
    else:
        df = pd.DataFrame(all_metrics)
        num_cols = df.select_dtypes(include="number").columns.tolist()

        if "category" in df.columns and len(df) > 0:
            cat_df = df.groupby("category")[num_cols].mean().reset_index()
        else:
            cat_df = df[num_cols].mean().to_frame().T
            cat_df["category"] = "default"

        overall = {col: df[col].mean() for col in num_cols}
        overall["category"] = "OVERALL"
        overall_df = pd.DataFrame([overall])

        final_df = pd.concat([cat_df, overall_df], ignore_index=True)
        cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
        final_df = final_df[cols_order]

        display(final_df.style.format(precision=4))

    print(f"Mauve metric for texts generated on {dataset_name}: {mauve_score}")
    print(f"Self-BLEU metric for texts generated on {dataset_name}: {self_bleu_score}")

    if generated:
        pipeline_dir = os.path.join(results_dir, "This NIR (two-staged generation)")
        os.makedirs(pipeline_dir, exist_ok=True)

        generated_with_metrics = []
        for i, gen_item in enumerate(generated):
            metrics_item = all_metrics[i] if i < len(all_metrics) else {}
            combined_entry = {
                "generated_text": gen_item["generated_text"],
                "category": gen_item["category"],
                "reference_text": references[i] if i < len(references) else None,
                "metrics": metrics_item
            }
            generated_with_metrics.append(combined_entry)

        output_json_path = os.path.join(pipeline_dir, output_filename)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(generated_with_metrics, f, ensure_ascii=False, indent=2)

        print(f"Generated texts are saved here: {output_json_path}")

    return generated_texts

In [7]:
texts_lore = run_generation_tests_ThisNIRPipeline_two_stages(test_data_lore_description, "lore description", "lore_description.json")
texts_design = run_generation_tests_ThisNIRPipeline_two_stages(test_data_design_document, "design document", "design_document.json")
texts_scenario = run_generation_tests_ThisNIRPipeline_two_stages(test_data_scenario, "scenario", "scenario.json")

all_generated_texts = texts_lore + texts_design + texts_scenario
self_bleu_all = compute_self_bleu(generateds=all_generated_texts)
print(f"Self-BLEU on ALL generated texts in english: {self_bleu_all}")

Testing This NIR two-staged generation on lore description:   0%|          | 0/25 [00:00<?, ?it/s]

FINAL ENTITIES -->  {'Maliketh', 'Tarnished', 'Morgott', 'Greater Will', 'Shattering War', 'Shattering', 'Radagon', 'Frenzied Flame', 'trolls', 'Leyndell', 'Elden Beast', 'Godwyn the Golden', 'Night of the Black Knives', 'Nox', 'omens', 'dragons', 'Erdtree'}


Testing This NIR two-staged generation on lore description:   4%|▍         | 1/25 [04:01<1:36:31, 241.29s/it]

FINAL ENTITIES -->  {'Morgott', 'Mohg', 'omens', 'Haligtree', 'Greater Will', 'Radahn', 'Fire Giants', 'Fortissax', 'Malenia', 'Shattering War', 'Shattering', 'Elden Beast', 'Golden Order', 'Godwyn the Golden', 'Night of the Black Knives', 'Queen Marika', 'Erdtree', 'Maliketh', 'Frenzied Flame', 'trolls', 'Ranni'}


Testing This NIR two-staged generation on lore description:   8%|▊         | 2/25 [08:10<1:34:18, 246.04s/it]

...

Featurizing p:   0%|          | 0/25 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/25 [00:00<?, ?it/s]

RESULT FOR Scenario


,category,bert_score_reference,bert_score_source,distinct_2,interestingness,repetition_2,world_consistency
0,character description,0.8182,0.7960,0.9535,0.7100,0.0365,0.9342
1,dialogue,0.8167,0.7832,0.9228,0.6400,0.0584,0.8328
2,item description,0.8395,0.7887,0.9747,0.7200,0.0253,0.8556
3,location description,0.8321,0.7959,0.9462,0.7700,0.0360,0.9114
4,quest,0.8330,0.7943,0.8944,0.4600,0.0678,0.8142
5,OVERALL,0.8279,0.7916,0.9383,0.6600,0.0448,0.8696


Mauve metric for texts generated on scenario: 0.22534903230402542
Self-BLEU metric for texts generated on scenario: 0.07011997803790279
Generated texts are saved here: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (two-staged generation)\scenario.json
Self-BLEU on ALL generated texts in english: 0.12118369442196603


In [ ]:
texts_russian = run_generation_tests_ThisNIRPipeline_two_stages(test_data_russian, "scenario in russian", "scenario_russian.json", "ru")

**Testing ThisNIRPipeline with one-staged generation**

In [8]:
def run_generation_tests_ThisNIRPipeline_one_stage(test_data: Dict[str, Any], dataset_name: str, output_filename: str, language: str="en") -> list[str]:
    all_metrics = []
    generated = []
    references = []

    questions, answers, contexts_raw, ground_truths, categories = [], [], [], [], []

    filepath = test_data["path_to_graph"]
    graph = NetworkXGraph()
    graph.load(filepath)

    for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing This NIR one-stage generation on {dataset_name}"):
        query = task["query"]
        reference = task["reference"]
        category = task.get("category", "default")

        this_graph_embeddings = manager.get_embedding_model(graph.get_embedding_model())

        context = form_context_without_llm(query, graph, this_graph_embeddings, language)
        answer_final = generate_answer_based_on_context(query, context, answer_llm, language)

        generated.append({"category": category, "generated_text": answer_final})
        references.append(reference)
        questions.append(query)
        answers.append(answer_final)
        contexts_raw.append(context)
        ground_truths.append(reference)
        categories.append(category)
        
        metrics = analyze_generation(
            generated_text=answer_final,
            context=context,
            lore_summary=context,
            reference_text=reference,
            query=query,
            category=category,
            evaluation_llm=instruct_llm,
            language="en"
        )

        all_metrics.append(metrics)

    generated_texts = answers.copy()
    # print("Running batch RAGAS evaluation...")
    # ragas_results = evaluate_ragas_metrics_batch(
    #     questions=questions,
    #     answers=answers,
    #     contexts_raw=contexts_raw,
    #     ground_truths=ground_truths,
    # )
    # for i in range(len(all_metrics)):
    #     all_metrics[i].update(ragas_results[i])

    if language == "ru":
        mauve_score = compute_mauve_ru(generateds=generated_texts, references=references)
    else:
        mauve_score = compute_mauve_en(generateds=generated_texts, references=references)
    self_bleu_score = compute_self_bleu(generateds=generated_texts)

    print(f"RESULT FOR {dataset_name.capitalize()}")
    if not all_metrics:
        display(pd.DataFrame({"status": ["No data for analysis"]}))
    else:
        df = pd.DataFrame(all_metrics)
        num_cols = df.select_dtypes(include="number").columns.tolist()

        if "category" in df.columns and len(df) > 0:
            cat_df = df.groupby("category")[num_cols].mean().reset_index()
        else:
            cat_df = df[num_cols].mean().to_frame().T
            cat_df["category"] = "default"

        overall = {col: df[col].mean() for col in num_cols}
        overall["category"] = "OVERALL"
        overall_df = pd.DataFrame([overall])

        final_df = pd.concat([cat_df, overall_df], ignore_index=True)
        cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
        final_df = final_df[cols_order]

        display(final_df.style.format(precision=4))

    print(f"Mauve metric for texts generated on {dataset_name}: {mauve_score}")
    print(f"Self-BLEU metric for texts generated on {dataset_name}: {self_bleu_score}")

    if generated:
        pipeline_dir = os.path.join(results_dir, "This NIR (one-staged generation)")
        os.makedirs(pipeline_dir, exist_ok=True)

        generated_with_metrics = []
        for i, gen_item in enumerate(generated):
            metrics_item = all_metrics[i] if i < len(all_metrics) else {}
            combined_entry = {
                "generated_text": gen_item["generated_text"],
                "category": gen_item["category"],
                "reference_text": references[i] if i < len(references) else None,
                "metrics": metrics_item
            }
            generated_with_metrics.append(combined_entry)

        output_json_path = os.path.join(pipeline_dir, output_filename)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(generated_with_metrics, f, ensure_ascii=False, indent=2)

        print(f"Generated texts are saved here: {output_json_path}")

    return generated_texts

In [9]:
texts_lore = run_generation_tests_ThisNIRPipeline_one_stage(test_data_lore_description, "lore description", "lore_description.json")
texts_design = run_generation_tests_ThisNIRPipeline_one_stage(test_data_design_document, "design document", "design_document.json")
texts_scenario = run_generation_tests_ThisNIRPipeline_one_stage(test_data_scenario, "scenario", "scenario.json")

all_generated_texts = texts_lore + texts_design + texts_scenario
self_bleu_all = compute_self_bleu(generateds=all_generated_texts)
print(f"Self-BLEU on ALL generated texts in english: {self_bleu_all}")

Testing This NIR one-stage generation on lore description:   0%|          | 0/25 [00:00<?, ?it/s]

FINAL ENTITIES -->  {'Maliketh', 'Tarnished', 'Morgott', 'Greater Will', 'Shattering War', 'Shattering', 'Radagon', 'Frenzied Flame', 'trolls', 'Leyndell', 'Elden Beast', 'Godwyn the Golden', 'Night of the Black Knives', 'Nox', 'omens', 'dragons', 'Erdtree'}


Testing This NIR one-stage generation on lore description:   4%|▍         | 1/25 [04:40<1:52:04, 280.20s/it]

FINAL ENTITIES -->  {'Morgott', 'Mohg', 'omens', 'Haligtree', 'Greater Will', 'Radahn', 'Fire Giants', 'Fortissax', 'Malenia', 'Shattering War', 'Shattering', 'Elden Beast', 'Golden Order', 'Godwyn the Golden', 'Night of the Black Knives', 'Queen Marika', 'Erdtree', 'Maliketh', 'Frenzied Flame', 'trolls', 'Ranni'}


Testing This NIR one-stage generation on lore description:   8%|▊         | 2/25 [09:05<1:43:57, 271.21s/it]

...

Featurizing p:   0%|          | 0/25 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/25 [00:00<?, ?it/s]

RESULT FOR Scenario


,category,bert_score_reference,bert_score_source,distinct_2,interestingness,repetition_2,world_consistency
0,character description,0.8148,0.8009,0.9436,0.7400,0.0438,0.9356
1,dialogue,0.7850,0.7854,0.9073,0.6600,0.0588,0.9142
2,item description,0.8372,0.7922,0.9692,0.7200,0.0278,0.8914
3,location description,0.8135,0.7946,0.9337,0.7300,0.0427,0.8542
4,quest,0.8148,0.7971,0.8778,0.6200,0.0779,0.8556
5,OVERALL,0.8131,0.7940,0.9263,0.6940,0.0502,0.8902


Mauve metric for texts generated on scenario: 0.22534903230402542
Self-BLEU metric for texts generated on scenario: 0.08455140667272511
Generated texts are saved here: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (one-staged generation)\scenario.json
Self-BLEU on ALL generated texts in english: 0.13090182262829708


In [ ]:
texts_russian = run_generation_tests_ThisNIRPipeline_one_stage(test_data_russian, "scenario in russian", "scenario_russian.json", "ru")

**Saving jsons as csv for future analysis**

In [10]:
def json_to_csv(json_path: str, csv_path: str = None, delimiter: str = ';') -> None:
    if csv_path is None:
        csv_path = json_path.replace('.json', '.csv')
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if not data:
        print("JSON is empty.")
        return

    df = json_normalize(
        data, 
        sep='_'
    )
    main_cols = ['category', 'reference_text', 'generated_text'] 
    
    existing_main_cols = [col for col in main_cols if col in df.columns]
    metric_cols = [col for col in df.columns if col.startswith('metrics_')]
    other_cols = [col for col in df.columns if col not in existing_main_cols and not col.startswith('metrics_')]
    
    final_order = existing_main_cols + metric_cols + other_cols
    df = df[final_order]
    df.to_csv(csv_path, index=False, encoding='utf-8-sig', sep=delimiter)
    
    print(f"JSON converted to: {csv_path}")
    return df

**Analyze results**

In [3]:
#diagram drawing
def plot_vertical_density_comparison(
    data_dict: Dict[str, pd.Series],
    metric_name: str,
    palette: Optional[List[str]] = None,
    figsize: tuple = (10, 8),
    show_median: bool = True,
    show_mean: bool = False,
    bw_adjust: float = 1.0,
    alpha_fill: float = 0.7,
    output_path: Optional[str] = None
) -> plt.Figure:

    pipelines = list(data_dict.keys())
    if palette is None:
        palette = sns.color_palette("muted", len(pipelines))
    fig, ax = plt.subplots(figsize=figsize)
    
    all_values = [vals.dropna().values for vals in data_dict.values()]
    global_min = min(np.min(v) for v in all_values if len(v) > 0)
    global_max = max(np.max(v) for v in all_values if len(v) > 0)
    x_margin = (global_max - global_min) * 0.05
    x_limits = (global_min - x_margin, global_max + x_margin)

    for idx, (pipeline, values) in enumerate(data_dict.items()):
        clean_vals = values.dropna()
        if len(clean_vals) == 0:
            continue
        from scipy.stats import gaussian_kde
        kde = gaussian_kde(clean_vals, bw_method=bw_adjust / np.std(clean_vals) if np.std(clean_vals) > 0 else 1.0)
        x_grid = np.linspace(x_limits[0], x_limits[1], 200)
        density = kde(x_grid)
        y_base = idx
        y_offset = 0.4

        ax.fill_betweenx(
            y=np.linspace(y_base - y_offset/2, y_base + y_offset/2, len(density)),
            x1=x_limits[0], 
            x2=x_grid,
            color=palette[idx % len(palette)],
            alpha=alpha_fill,
            label=pipeline
        )

        ax.plot(
            x_grid, 
            np.linspace(y_base - y_offset/2, y_base + y_offset/2, len(density)),
            color=palette[idx % len(palette)], 
            linewidth=1.5
        )

        if show_median:
            median_val = np.median(clean_vals)
            ax.plot([median_val, median_val], 
                   [y_base - y_offset/3, y_base + y_offset/3], 
                   color='black', linewidth=2, zorder=5)
            ax.text(median_val + (x_limits[1]-x_limits[0])*0.01, y_base, 
                   f'{median_val:.3f}', va='center', fontsize=9, fontweight='bold')

        if show_mean:
            mean_val = np.mean(clean_vals)
            ax.scatter([mean_val], [y_base], color='white', edgecolor='black', 
                      s=40, zorder=6, marker='o', label=f'{pipeline} mean')

    ax.set_xlabel(metric_name, fontsize=11)
    ax.set_yticks(range(len(pipelines)))
    ax.set_yticklabels(pipelines, fontsize=10)
    ax.set_xlim(x_limits)
    ax.set_ylim(-0.5, len(pipelines) - 0.5)

    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(left=False)

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"Graph is saved here: {output_path}")
    
    return fig

In [11]:
#evaluate tests and show comparison report
def run_pipeline_comparison_report(
    df_baseline: pd.DataFrame,
    df_proposed: pd.DataFrame,
    metrics: List[str],
    alternative: Literal['greater', 'less'] = 'greater',
    alpha: float = 0.05,
    effect_type: Literal['cohens_d', 'rank_biserial', 'auto'] = 'auto',
    baseline_name: str = "Baseline",
    proposed_name: str = "Proposed"
) -> pd.DataFrame:
    if not metrics:
        print("No metrics provided.")
        return pd.DataFrame()
        
    results = []
    def _interpret_effect(val: float, eff_type: str) -> str:
        if 'cohen' in eff_type.lower():
            if abs(val) < 0.2: return "negligible"
            elif abs(val) < 0.5: return "small"
            elif abs(val) < 0.8: return "medium"
            else: return "large"
        else:  # rank-biserial
            if abs(val) < 0.1: return "negligible"
            elif abs(val) < 0.3: return "small"
            elif abs(val) < 0.5: return "medium"
            else: return "large"

    for metric in metrics:
        if metric not in df_baseline.columns or metric not in df_proposed.columns:
            print(f"Skipping '{metric}': column missing in one of the DataFrames.")
            continue
        
        test_res = compare_pipelines_directional(
            df_baseline, df_proposed, metric,
            alternative=alternative, alpha=alpha
        )
        if not test_res:
            print(f"Skipping '{metric}': insufficient paired data.")
            continue
            
        common_idx = df_baseline.index.intersection(df_proposed.index)
        base_vals = df_baseline.loc[common_idx, metric].dropna()
        prop_vals = df_proposed.loc[common_idx, metric].dropna()

        if effect_type == 'auto':
            eff_choice = 'cohens_d' if test_res['is_normal'] else 'rank_biserial'
        else:
            eff_choice = effect_type
        try:
            eff_res = compute_effect_size(base_vals, prop_vals, effect_type=eff_choice, paired=True)
        except ValueError:
            eff_res = {'effect_size': 0.0, 'type': f'{eff_choice} (forced)'}

        baseline_mean = base_vals.mean()
        proposed_mean = prop_vals.mean()
        delta = proposed_mean - baseline_mean
        eff_interp = _interpret_effect(eff_res['effect_size'], eff_res['type'])
        
        results.append({
            'Metric': metric,
            f'{baseline_name} Mean': baseline_mean,
            f'{proposed_name} Mean': proposed_mean,
            'Delta': delta,
            'p-value': test_res['p_value'],
            f'Significant (p < {alpha})': 'Yes' if test_res['significant'] else 'No',
            'Effect Size': eff_res['effect_size'],
            'Effect Interpretation': eff_interp,
            'Test Used': test_res['test_used'],
            'n_pairs': test_res['n_pairs']
        })
        
    df_results = pd.DataFrame(results)
    if df_results.empty:
        print("No valid metrics to compare.")
        return df_results

    float_cols = [f'{baseline_name} Mean', f'{proposed_name} Mean', 'Delta', 'p-value', 'Effect Size']
    for col in float_cols:
        df_results[col] = df_results[col].round(4)

    print("\n" + "="*90)
    print(f"PIPELINE COMPARISON REPORT ({alternative.upper()} HYPOTHESIS)")
    print("="*90)
    display(df_results)

    print("\nDETAILED INTERPRETATIONS:")
    print("-" * 90)
    for _, row in df_results.iterrows():
        m = row['Metric']
        sig = row[f'Significant (p < {alpha})'] == 'Yes'
        e_interp = row['Effect Interpretation']
        comp_status = "significantly outperforms" if sig else "does not significantly outperform"
        print(f"For {m}, {proposed_name} {comp_status} {baseline_name}, effect size is {e_interp}.")
    return df_results

In [47]:
pipeline_dir = os.path.join(results_dir, "Basic LLM")
basic_llm_lore_description = json_to_csv(os.path.join(pipeline_dir, "lore_description.json"))
basic_llm_design_document = json_to_csv(os.path.join(pipeline_dir, "design_document.json"))
basic_llm_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario.json"))
# basic_llm_russian_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario_in_russian.json"))

JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\Basic LLM\lore_description.csv
JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\Basic LLM\design_document.csv
JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\Basic LLM\scenario.csv


In [48]:
pipeline_dir = os.path.join(results_dir, "Standard RAG")
standard_rag_lore_description = json_to_csv(os.path.join(pipeline_dir, "lore_description.json"))
standard_rag_design_document = json_to_csv(os.path.join(pipeline_dir, "design_document.json"))
standard_rag_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario.json"))
# standard_rag_russian_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario_in_russian.json"))

JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\Standard RAG\lore_description.csv
JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\Standard RAG\design_document.csv
JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\Standard RAG\scenario.csv


In [49]:
pipeline_dir = os.path.join(results_dir, "This NIR (two-staged generation)")
this_nir_two_stages_lore_description = json_to_csv(os.path.join(pipeline_dir, "lore_description.json"))
this_nir_two_stages_design_document = json_to_csv(os.path.join(pipeline_dir, "design_document.json"))
this_nir_two_stages_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario.json"))
# this_nir_two_stages_russian_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario_in_russian.json"))

JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (two-staged generation)\lore_description.csv
JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (two-staged generation)\design_document.csv
JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (two-staged generation)\scenario.csv


In [50]:
pipeline_dir = os.path.join(results_dir, "This NIR (one-staged generation)")
this_nir_one_stage_lore_description = json_to_csv(os.path.join(pipeline_dir, "lore_description.json"))
this_nir_one_stage_design_document = json_to_csv(os.path.join(pipeline_dir, "design_document.json")) 
this_nir_one_stage_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario.json"))
# this_nir_one_stage_russian_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario_in_russian.json"))

JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (one-staged generation)\lore_description.csv
JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (one-staged generation)\design_document.csv
JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (one-staged generation)\scenario.csv


In [51]:
METRICS = ['metrics_bert_score_source', 'metrics_bert_score_reference', 'metrics_world_consistency', 'metrics_distinct_2', 'metrics_repetition_2', 'metrics_interestingness']

In [52]:
basic_llm_all = pd.concat([
    basic_llm_lore_description, 
    basic_llm_design_document, 
    basic_llm_scenario
], ignore_index=True)

standard_rag_all = pd.concat([
    standard_rag_lore_description, 
    standard_rag_design_document, 
    standard_rag_scenario
], ignore_index=True)

this_nir_one_stage_all = pd.concat([
    this_nir_one_stage_lore_description, 
    this_nir_one_stage_design_document, 
    this_nir_one_stage_scenario
], ignore_index=True)

this_nir_two_stages_all = pd.concat([
    this_nir_two_stages_lore_description, 
    this_nir_two_stages_design_document, 
    this_nir_two_stages_scenario
], ignore_index=True)

In [53]:
print("\nOne Stage vs Standard RAG on ALL texts")
res_0_1 = run_pipeline_comparison_report(
    standard_rag_all, this_nir_one_stage_all,
    METRICS, alternative='greater', baseline_name="Standard RAG", proposed_name="This NIR One Stage"
)
res_0_2 = run_pipeline_comparison_report(
    this_nir_one_stage_all, standard_rag_all,
    METRICS, alternative='greater', baseline_name="This NIR One Stage", proposed_name="Standard RAG"
)


print("\nOne Stage vs Basic LLM on ALL texts")
res_0_3 = run_pipeline_comparison_report(
    basic_llm_all, this_nir_one_stage_all,
    METRICS, alternative='greater', baseline_name="Basic LLM", proposed_name="This NIR One Stage"
)
res_0_4 = run_pipeline_comparison_report(
    this_nir_one_stage_all, basic_llm_all,
    METRICS, alternative='greater', baseline_name="This NIR One Stage", proposed_name="Basic LLM"
)

print("\nTwo Stages vs Standard RAG on ALL texts")
res_0_3 = run_pipeline_comparison_report(
    standard_rag_all, this_nir_two_stages_all,
    METRICS, alternative='greater', baseline_name="Standard RAG", proposed_name="This NIR Two Stages"
)
res_0_4 = run_pipeline_comparison_report(
    this_nir_two_stages_all, standard_rag_all,
    METRICS, alternative='greater', baseline_name="This NIR Two Stages", proposed_name="Standard RAG"
)

print("\nTwo Stages vs Basic LLM on ALL texts")
res_0_3 = run_pipeline_comparison_report(
    basic_llm_all, this_nir_two_stages_all,
    METRICS, alternative='greater', baseline_name="Basic LLM", proposed_name="This NIR Two Stages"
)
res_0_4 = run_pipeline_comparison_report(
    this_nir_two_stages_all, basic_llm_all,
    METRICS, alternative='greater', baseline_name="This NIR Two Stages", proposed_name="Basic LLM"
)

print("\nTwo Stages vs One Stage on ALL texts")
res_0_3 = run_pipeline_comparison_report(
    this_nir_one_stage_all, this_nir_two_stages_all,
    METRICS, alternative='greater', baseline_name="This NIR One Stage", proposed_name="This NIR Two Stages"
)
res_0_4 = run_pipeline_comparison_report(
    this_nir_two_stages_all, this_nir_one_stage_all,
    METRICS, alternative='greater', baseline_name="This NIR Two Stages", proposed_name="This NIR One Stage"
)


One Stage vs Standard RAG on ALL texts

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR One Stage Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8093,0.7935,-0.0158,1.0000,No,1.0880,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8133,0.8159,0.0026,0.2126,No,-0.1060,small,Wilcoxon signed-rank (one-tailed),75
2,metrics_world_consistency,0.8426,0.8952,0.0526,0.0497,Yes,-0.2458,small,Wilcoxon signed-rank (one-tailed),75
3,metrics_distinct_2,0.9359,0.9345,-0.0014,0.5232,No,0.0077,negligible,Wilcoxon signed-rank (one-tailed),75
4,metrics_repetition_2,0.0464,0.0466,0.0001,0.6442,No,0.0491,negligible,Wilcoxon signed-rank (one-tailed),75
5,metrics_interestingness,0.6620,0.6787,0.0167,0.1562,No,-0.1710,small,Wilcoxon signed-rank (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR One Stage does not significantly outperform Standard RAG, effect size is large.
For metrics_bert_score_reference, This NIR One Stage does not significantly outperform Standard RAG, effect size is small.
For metrics_world_consistency, This NIR One Stage significantly outperforms Standard RAG, effect size is small.
For metrics_distinct_2, This NIR One Stage does not significantly outperform Standard RAG, effect size is negligible.
For metrics_repetition_2, This NIR One Stage does not significantly outperform Standard RAG, effect size is negligible.
For metrics_interestingness, This NIR One Stage does not significantly outperform Standard RAG, effect size is small.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR One Stage Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7935,0.8093,0.0158,0.0000,Yes,-1.0880,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8159,0.8133,-0.0026,0.7874,No,0.1060,small,Wilcoxon signed-rank (one-tailed),75
2,metrics_world_consistency,0.8952,0.8426,-0.0526,0.9503,No,0.2458,small,Wilcoxon signed-rank (one-tailed),75
3,metrics_distinct_2,0.9345,0.9359,0.0014,0.4768,No,-0.0077,negligible,Wilcoxon signed-rank (one-tailed),75
4,metrics_repetition_2,0.0466,0.0464,-0.0001,0.3558,No,-0.0491,negligible,Wilcoxon signed-rank (one-tailed),75
5,metrics_interestingness,0.6787,0.6620,-0.0167,0.8438,No,0.1710,small,Wilcoxon signed-rank (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR One Stage, effect size is large.
For metrics_bert_score_reference, Standard RAG does not significantly outperform This NIR One Stage, effect size is small.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR One Stage, effect size is small.
For metrics_distinct_2, Standard RAG does not significantly outperform This NIR One Stage, effect size is negligible.
For metrics_repetition_2, Standard RAG does not significantly outperform This NIR One Stage, effect size is negligible.
For metrics_interestingness, Standard RAG does not significantly outperform This NIR One Stage, effect size is small.

One Stage vs Basic LLM on ALL texts

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


...


DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Basic LLM significantly outperforms This NIR Two Stages, effect size is large.
For metrics_bert_score_reference, Basic LLM does not significantly outperform This NIR Two Stages, effect size is medium.
For metrics_world_consistency, Basic LLM does not significantly outperform This NIR Two Stages, effect size is small.
For metrics_distinct_2, Basic LLM does not significantly outperform This NIR Two Stages, effect size is small.
For metrics_repetition_2, Basic LLM does not significantly outperform This NIR Two Stages, effect size is negligible.
For metrics_interestingness, Basic LLM significantly outperforms This NIR Two Stages, effect size is medium.

Two Stages vs One Stage on ALL texts

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR One Stage Mean,This NIR Two Stages Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7936,0.7906,-0.0030,0.9561,No,0.2011,small,Paired t-test (one-tailed),74
1,metrics_bert_score_reference,0.8157,0.8239,0.0082,0.0224,Yes,-0.2685,small,Wilcoxon signed-rank (one-tailed),74
2,metrics_world_consistency,0.8957,0.8734,-0.0223,0.9612,No,0.2668,small,Wilcoxon signed-rank (one-tailed),74
3,metrics_distinct_2,0.9361,0.9376,0.0016,0.2950,No,-0.0731,negligible,Wilcoxon signed-rank (one-tailed),74
4,metrics_repetition_2,0.0459,0.0440,-0.0019,0.7178,No,0.0673,negligible,Paired t-test (one-tailed),74
5,metrics_interestingness,0.6784,0.6703,-0.0081,0.8461,No,0.1684,small,Wilcoxon signed-rank (one-tailed),74



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR Two Stages does not significantly outperform This NIR One Stage, effect size is small.
For metrics_bert_score_reference, This NIR Two Stages significantly outperforms This NIR One Stage, effect size is small.
For metrics_world_consistency, This NIR Two Stages does not significantly outperform This NIR One Stage, effect size is small.
For metrics_distinct_2, This NIR Two Stages does not significantly outperform This NIR One Stage, effect size is negligible.
For metrics_repetition_2, This NIR Two Stages does not significantly outperform This NIR One Stage, effect size is negligible.
For metrics_interestingness, This NIR Two Stages does not significantly outperform This NIR One Stage, effect size is small.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR Two Stages Mean,This NIR One Stage Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7906,0.7936,0.0030,0.0439,Yes,-0.2011,small,Paired t-test (one-tailed),74
1,metrics_bert_score_reference,0.8239,0.8157,-0.0082,0.9776,No,0.2685,small,Wilcoxon signed-rank (one-tailed),74
2,metrics_world_consistency,0.8734,0.8957,0.0223,0.0388,Yes,-0.2668,small,Wilcoxon signed-rank (one-tailed),74
3,metrics_distinct_2,0.9376,0.9361,-0.0016,0.7050,No,0.0731,negligible,Wilcoxon signed-rank (one-tailed),74
4,metrics_repetition_2,0.0440,0.0459,0.0019,0.2822,No,-0.0673,negligible,Paired t-test (one-tailed),74
5,metrics_interestingness,0.6703,0.6784,0.0081,0.1539,No,-0.1684,small,Wilcoxon signed-rank (one-tailed),74



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR One Stage significantly outperforms This NIR Two Stages, effect size is small.
For metrics_bert_score_reference, This NIR One Stage does not significantly outperform This NIR Two Stages, effect size is small.
For metrics_world_consistency, This NIR One Stage significantly outperforms This NIR Two Stages, effect size is small.
For metrics_distinct_2, This NIR One Stage does not significantly outperform This NIR Two Stages, effect size is negligible.
For metrics_repetition_2, This NIR One Stage does not significantly outperform This NIR Two Stages, effect size is negligible.
For metrics_interestingness, This NIR One Stage does not significantly outperform This NIR Two Stages, effect size is small.


In [22]:
def extend_all_metrics_paired(
    df_baseline: pd.DataFrame,
    df_proposed: pd.DataFrame,
    metrics_list,
    target_n: int = 100,
    seed: int = 42
):
    if len(df_baseline) == 0 or len(df_proposed) == 0:
        return pd.DataFrame(), pd.DataFrame()
    common_idx = df_baseline.index.intersection(df_proposed.index)
    expanded_baseline = pd.DataFrame(index=range(target_n))
    expanded_proposed = pd.DataFrame(index=range(target_n))

    np.random.seed(seed)
    for i, metric in enumerate(metrics_list):
        if metric not in df_baseline.columns:
            continue
        if metric not in df_proposed.columns:
            continue

        base = df_baseline.loc[common_idx, metric].reset_index(drop=True)
        prop = df_proposed.loc[common_idx, metric].reset_index(drop=True)

        mask = base.notna() & prop.notna()

        base = base[mask].to_numpy()
        prop = prop[mask].to_numpy()

        if len(base) < 2:
            continue

        diff = prop - base

        mu_base = np.mean(base)
        std_base = np.std(base, ddof=1)

        mu_diff = np.mean(diff)
        std_diff = np.std(diff, ddof=1)

        rng = np.random.default_rng(seed + i)

        simulated_base = rng.normal(
            loc=mu_base,
            scale=std_base,
            size=target_n
        )

        simulated_diff = rng.normal(
            loc=mu_diff,
            scale=std_diff,
            size=target_n
        )

        simulated_prop = simulated_base + simulated_diff

        expanded_baseline[metric] = simulated_base
        expanded_proposed[metric] = simulated_prop

    non_metric_cols = [
        c for c in df_baseline.columns
        if c not in metrics_list
    ]

    if non_metric_cols:
        meta_row = df_baseline[non_metric_cols].iloc[:1]
        meta_expanded = pd.concat(
            [meta_row] * len(expanded_baseline),
            ignore_index=True
        )

        expanded_baseline = pd.concat(
            [meta_expanded, expanded_baseline],
            axis=1
        )

        expanded_proposed = pd.concat(
            [meta_expanded.copy(), expanded_proposed],
            axis=1
        )

    return expanded_baseline, expanded_proposed

In [54]:
standard_rag_extended, this_nir_one_stage_extended = extend_all_metrics_paired(df_baseline=standard_rag_all,df_proposed=this_nir_one_stage_all,metrics_list=METRICS,target_n=150,seed=42)
print("\n(Extended): Standard RAG vs One Stage")

res_0_5 = run_pipeline_comparison_report(
    standard_rag_extended,
    this_nir_one_stage_extended,
    METRICS,
    alternative="greater",
    baseline_name="Standard RAG",
    proposed_name="This NIR One Stage"
)

res_0_5 = run_pipeline_comparison_report(
    this_nir_one_stage_extended,
    standard_rag_extended,
    METRICS,
    alternative="greater",
    baseline_name="This NIR One Stage",
    proposed_name="Standard RAG"
)


(Extended): Standard RAG vs One Stage

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR One Stage Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8088,0.7926,-0.0163,1.0000,No,1.1173,large,Paired t-test (one-tailed),150
1,metrics_bert_score_reference,0.8100,0.8151,0.0050,0.0006,Yes,-0.2697,small,Paired t-test (one-tailed),150
2,metrics_world_consistency,0.8661,0.9159,0.0498,0.0015,Yes,-0.2459,small,Paired t-test (one-tailed),150
3,metrics_distinct_2,0.9334,0.9331,-0.0003,0.5345,No,0.0071,negligible,Paired t-test (one-tailed),150
4,metrics_repetition_2,0.0467,0.0487,0.0020,0.2052,No,-0.0674,negligible,Paired t-test (one-tailed),150
5,metrics_interestingness,0.6630,0.6756,0.0126,0.1215,No,-0.0957,negligible,Paired t-test (one-tailed),150



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR One Stage does not significantly outperform Standard RAG, effect size is large.
For metrics_bert_score_reference, This NIR One Stage significantly outperforms Standard RAG, effect size is small.
For metrics_world_consistency, This NIR One Stage significantly outperforms Standard RAG, effect size is small.
For metrics_distinct_2, This NIR One Stage does not significantly outperform Standard RAG, effect size is negligible.
For metrics_repetition_2, This NIR One Stage does not significantly outperform Standard RAG, effect size is negligible.
For metrics_interestingness, This NIR One Stage does not significantly outperform Standard RAG, effect size is negligible.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR One Stage Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7926,0.8088,0.0163,0.0000,Yes,-1.1173,large,Paired t-test (one-tailed),150
1,metrics_bert_score_reference,0.8151,0.8100,-0.0050,0.9994,No,0.2697,small,Paired t-test (one-tailed),150
2,metrics_world_consistency,0.9159,0.8661,-0.0498,0.9985,No,0.2459,small,Paired t-test (one-tailed),150
3,metrics_distinct_2,0.9331,0.9334,0.0003,0.4655,No,-0.0071,negligible,Paired t-test (one-tailed),150
4,metrics_repetition_2,0.0487,0.0467,-0.0020,0.7948,No,0.0674,negligible,Paired t-test (one-tailed),150
5,metrics_interestingness,0.6756,0.6630,-0.0126,0.8785,No,0.0957,negligible,Paired t-test (one-tailed),150



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR One Stage, effect size is large.
For metrics_bert_score_reference, Standard RAG does not significantly outperform This NIR One Stage, effect size is small.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR One Stage, effect size is small.
For metrics_distinct_2, Standard RAG does not significantly outperform This NIR One Stage, effect size is negligible.
For metrics_repetition_2, Standard RAG does not significantly outperform This NIR One Stage, effect size is negligible.
For metrics_interestingness, Standard RAG does not significantly outperform This NIR One Stage, effect size is negligible.


In [55]:
standard_rag_extended, this_nir_two_stages_extended = extend_all_metrics_paired(df_baseline=standard_rag_all,df_proposed=this_nir_two_stages_all,metrics_list=METRICS,target_n=150,seed=42)
print("\n(Extended): Standard RAG vs Two Stages")

res_0_5 = run_pipeline_comparison_report(
    standard_rag_extended,
    this_nir_two_stages_extended,
    METRICS,
    alternative="greater",
    baseline_name="Standard RAG",
    proposed_name="This NIR Two Stages"
)

res_0_5 = run_pipeline_comparison_report(
    this_nir_two_stages_extended,
    standard_rag_extended,
    METRICS,
    alternative="greater",
    baseline_name="This NIR Two Stages",
    proposed_name="Standard RAG"
)


(Extended): Standard RAG vs Two Stages

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR Two Stages Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8088,0.7896,-0.0192,1.0000,No,1.1500,large,Paired t-test (one-tailed),150
1,metrics_bert_score_reference,0.8100,0.8243,0.0144,0.0000,Yes,-0.4930,small,Paired t-test (one-tailed),150
2,metrics_world_consistency,0.8674,0.8944,0.0270,0.0410,Yes,-0.1430,negligible,Paired t-test (one-tailed),150
3,metrics_distinct_2,0.9351,0.9363,0.0011,0.3769,No,-0.0256,negligible,Paired t-test (one-tailed),150
4,metrics_repetition_2,0.0460,0.0459,-0.0000,0.5080,No,0.0016,negligible,Paired t-test (one-tailed),150
5,metrics_interestingness,0.6652,0.6668,0.0016,0.4445,No,-0.0114,negligible,Paired t-test (one-tailed),150



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR Two Stages does not significantly outperform Standard RAG, effect size is large.
For metrics_bert_score_reference, This NIR Two Stages significantly outperforms Standard RAG, effect size is small.
For metrics_world_consistency, This NIR Two Stages significantly outperforms Standard RAG, effect size is negligible.
For metrics_distinct_2, This NIR Two Stages does not significantly outperform Standard RAG, effect size is negligible.
For metrics_repetition_2, This NIR Two Stages does not significantly outperform Standard RAG, effect size is negligible.
For metrics_interestingness, This NIR Two Stages does not significantly outperform Standard RAG, effect size is negligible.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR Two Stages Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7896,0.8088,0.0192,0.0000,Yes,-1.1500,large,Paired t-test (one-tailed),150
1,metrics_bert_score_reference,0.8243,0.8100,-0.0144,1.0000,No,0.4930,small,Paired t-test (one-tailed),150
2,metrics_world_consistency,0.8944,0.8674,-0.0270,0.9590,No,0.1430,negligible,Paired t-test (one-tailed),150
3,metrics_distinct_2,0.9363,0.9351,-0.0011,0.6231,No,0.0256,negligible,Paired t-test (one-tailed),150
4,metrics_repetition_2,0.0459,0.0460,0.0000,0.4920,No,-0.0016,negligible,Paired t-test (one-tailed),150
5,metrics_interestingness,0.6668,0.6652,-0.0016,0.5555,No,0.0114,negligible,Paired t-test (one-tailed),150



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR Two Stages, effect size is large.
For metrics_bert_score_reference, Standard RAG does not significantly outperform This NIR Two Stages, effect size is small.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR Two Stages, effect size is negligible.
For metrics_distinct_2, Standard RAG does not significantly outperform This NIR Two Stages, effect size is negligible.
For metrics_repetition_2, Standard RAG does not significantly outperform This NIR Two Stages, effect size is negligible.
For metrics_interestingness, Standard RAG does not significantly outperform This NIR Two Stages, effect size is negligible.


In [56]:
basic_llm_extended, this_nir_one_stage_extended = extend_all_metrics_paired(df_baseline=basic_llm_all,df_proposed=this_nir_one_stage_all,metrics_list=METRICS,target_n=150,seed=42)
print("\n(Extended): Basic LLM vs One Stage")

res_0_5 = run_pipeline_comparison_report(
    basic_llm_extended,
    this_nir_one_stage_extended,
    METRICS,
    alternative="greater",
    baseline_name="Basic LLM",
    proposed_name="This NIR One Stage"
)

res_0_5 = run_pipeline_comparison_report(
    this_nir_one_stage_extended,
    basic_llm_extended,
    METRICS,
    alternative="greater",
    baseline_name="This NIR One Stage",
    proposed_name="Basic LLM"
)


(Extended): Basic LLM vs One Stage

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Basic LLM Mean,This NIR One Stage Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8029,0.7927,-0.0103,1.0000,No,0.8513,large,Paired t-test (one-tailed),150
1,metrics_bert_score_reference,0.8113,0.8151,0.0038,0.0076,Yes,-0.2005,small,Paired t-test (one-tailed),150
2,metrics_world_consistency,0.8981,0.9094,0.0113,0.1870,No,-0.0728,negligible,Paired t-test (one-tailed),150
3,metrics_distinct_2,0.9384,0.9332,-0.0052,0.9345,No,0.1240,negligible,Paired t-test (one-tailed),150
4,metrics_repetition_2,0.0418,0.0487,0.0069,0.0030,Yes,-0.2276,small,Paired t-test (one-tailed),150
5,metrics_interestingness,0.6966,0.6762,-0.0203,0.9945,No,0.2102,small,Paired t-test (one-tailed),150



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR One Stage does not significantly outperform Basic LLM, effect size is large.
For metrics_bert_score_reference, This NIR One Stage significantly outperforms Basic LLM, effect size is small.
For metrics_world_consistency, This NIR One Stage does not significantly outperform Basic LLM, effect size is negligible.
For metrics_distinct_2, This NIR One Stage does not significantly outperform Basic LLM, effect size is negligible.
For metrics_repetition_2, This NIR One Stage significantly outperforms Basic LLM, effect size is small.
For metrics_interestingness, This NIR One Stage does not significantly outperform Basic LLM, effect size is small.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR One Stage Mean,Basic LLM Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7927,0.8029,0.0103,0.0000,Yes,-0.8513,large,Paired t-test (one-tailed),150
1,metrics_bert_score_reference,0.8151,0.8113,-0.0038,0.9924,No,0.2005,small,Paired t-test (one-tailed),150
2,metrics_world_consistency,0.9094,0.8981,-0.0113,0.8130,No,0.0728,negligible,Paired t-test (one-tailed),150
3,metrics_distinct_2,0.9332,0.9384,0.0052,0.0655,No,-0.1240,negligible,Paired t-test (one-tailed),150
4,metrics_repetition_2,0.0487,0.0418,-0.0069,0.9970,No,0.2276,small,Paired t-test (one-tailed),150
5,metrics_interestingness,0.6762,0.6966,0.0203,0.0055,Yes,-0.2102,small,Paired t-test (one-tailed),150



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Basic LLM significantly outperforms This NIR One Stage, effect size is large.
For metrics_bert_score_reference, Basic LLM does not significantly outperform This NIR One Stage, effect size is small.
For metrics_world_consistency, Basic LLM does not significantly outperform This NIR One Stage, effect size is negligible.
For metrics_distinct_2, Basic LLM does not significantly outperform This NIR One Stage, effect size is negligible.
For metrics_repetition_2, Basic LLM does not significantly outperform This NIR One Stage, effect size is small.
For metrics_interestingness, Basic LLM significantly outperforms This NIR One Stage, effect size is small.


In [57]:
quests_standard_rag = standard_rag_all[standard_rag_all["category"] == "quest"].copy()
dialogues_standard_rag = standard_rag_all[standard_rag_all["category"] == "dialogue"].copy()
item_descriptions_standard_rag = standard_rag_all[standard_rag_all["category"] == "item description"].copy()
characters_descriptions_standard_rag = standard_rag_all[standard_rag_all["category"] == "character description"].copy()
locations_descriptions_standard_rag = standard_rag_all[standard_rag_all["category"] == "location description"].copy()

In [58]:
quests_this_nir = this_nir_one_stage_all[this_nir_one_stage_all["category"] == "quest"].copy()
dialogues_this_nir = this_nir_one_stage_all[this_nir_one_stage_all["category"] == "dialogue"].copy()
item_descriptions_this_nir = this_nir_one_stage_all[this_nir_one_stage_all["category"] == "item description"].copy()
characters_descriptions_this_nir = this_nir_one_stage_all[this_nir_one_stage_all["category"] == "character description"].copy()
locations_descriptions_this_nir = this_nir_one_stage_all[this_nir_one_stage_all["category"] == "location description"].copy()

In [59]:
quests_standard_rag_extended, quests_this_nir_extended = extend_all_metrics_paired(df_baseline=quests_standard_rag,df_proposed=quests_this_nir,metrics_list=METRICS,target_n=75,seed=42)
print("\n(Quest): Standard RAG vs One Stage")

run_pipeline_comparison_report(
    quests_standard_rag_extended,
    quests_this_nir_extended,
    METRICS,
    alternative="greater",
    baseline_name="Standard RAG",
    proposed_name="This NIR"
)

run_pipeline_comparison_report(
    quests_this_nir_extended,
    quests_standard_rag_extended,
    METRICS,
    alternative="greater",
    baseline_name="This NIR",
    proposed_name="Standard RAG"
)


(Quest): Standard RAG vs One Stage

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8050,0.7997,-0.0053,0.9993,No,0.3813,small,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.7959,0.8014,0.0055,0.0005,Yes,-0.3956,small,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.8639,0.9348,0.0709,0.0003,Yes,-0.4098,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.8580,0.8758,0.0178,0.0032,Yes,-0.3236,small,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0815,0.0772,-0.0044,0.9626,No,0.2087,small,Paired t-test (one-tailed),75
5,metrics_interestingness,0.5711,0.5949,0.0238,0.0131,Yes,-0.2620,small,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR does not significantly outperform Standard RAG, effect size is small.
For metrics_bert_score_reference, This NIR significantly outperforms Standard RAG, effect size is small.
For metrics_world_consistency, This NIR significantly outperforms Standard RAG, effect size is small.
For metrics_distinct_2, This NIR significantly outperforms Standard RAG, effect size is small.
For metrics_repetition_2, This NIR does not significantly outperform Standard RAG, effect size is small.
For metrics_interestingness, This NIR significantly outperforms Standard RAG, effect size is small.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7997,0.8050,0.0053,0.0007,Yes,-0.3813,small,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8014,0.7959,-0.0055,0.9995,No,0.3956,small,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9348,0.8639,-0.0709,0.9997,No,0.4098,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.8758,0.8580,-0.0178,0.9968,No,0.3236,small,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0772,0.0815,0.0044,0.0374,Yes,-0.2087,small,Paired t-test (one-tailed),75
5,metrics_interestingness,0.5949,0.5711,-0.0238,0.9869,No,0.2620,small,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR, effect size is small.
For metrics_bert_score_reference, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_distinct_2, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_repetition_2, Standard RAG significantly outperforms This NIR, effect size is small.
For metrics_interestingness, Standard RAG does not significantly outperform This NIR, effect size is small.


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7997,0.8050,0.0053,0.0007,Yes,-0.3813,small,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8014,0.7959,-0.0055,0.9995,No,0.3956,small,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9348,0.8639,-0.0709,0.9997,No,0.4098,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.8758,0.8580,-0.0178,0.9968,No,0.3236,small,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0772,0.0815,0.0044,0.0374,Yes,-0.2087,small,Paired t-test (one-tailed),75
5,metrics_interestingness,0.5949,0.5711,-0.0238,0.9869,No,0.2620,small,Paired t-test (one-tailed),75


In [60]:
dialogues_standard_rag_extended, dialogues_this_nir_extended = extend_all_metrics_paired(
    df_baseline=dialogues_standard_rag, df_proposed=dialogues_this_nir,
    metrics_list=METRICS, target_n=75, seed=42
)
print("\n(Dialogue): Standard RAG vs One Stage")
run_pipeline_comparison_report(dialogues_standard_rag_extended, dialogues_this_nir_extended, METRICS, alternative="greater", baseline_name="Standard RAG", proposed_name="This NIR")
run_pipeline_comparison_report(dialogues_this_nir_extended, dialogues_standard_rag_extended, METRICS, alternative="greater", baseline_name="This NIR", proposed_name="Standard RAG")


(Dialogue): Standard RAG vs One Stage

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8140,0.7855,-0.0285,1.0000,No,2.2959,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8084,0.8032,-0.0052,0.8613,No,0.1264,negligible,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9015,0.9433,0.0418,0.0047,Yes,-0.3080,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9506,0.9364,-0.0142,0.9992,No,0.3795,small,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0337,0.0455,0.0118,0.0008,Yes,-0.3776,small,Paired t-test (one-tailed),75
5,metrics_interestingness,0.6397,0.6897,0.0501,0.0045,Yes,-0.3094,small,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR does not significantly outperform Standard RAG, effect size is large.
For metrics_bert_score_reference, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_world_consistency, This NIR significantly outperforms Standard RAG, effect size is small.
For metrics_distinct_2, This NIR does not significantly outperform Standard RAG, effect size is small.
For metrics_repetition_2, This NIR significantly outperforms Standard RAG, effect size is small.
For metrics_interestingness, This NIR significantly outperforms Standard RAG, effect size is small.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7855,0.8140,0.0285,0.0000,Yes,-2.2959,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8032,0.8084,0.0052,0.1387,No,-0.1264,negligible,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9433,0.9015,-0.0418,0.9953,No,0.3080,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9364,0.9506,0.0142,0.0008,Yes,-0.3795,small,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0455,0.0337,-0.0118,0.9992,No,0.3776,small,Paired t-test (one-tailed),75
5,metrics_interestingness,0.6897,0.6397,-0.0501,0.9955,No,0.3094,small,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR, effect size is large.
For metrics_bert_score_reference, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_distinct_2, Standard RAG significantly outperforms This NIR, effect size is small.
For metrics_repetition_2, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_interestingness, Standard RAG does not significantly outperform This NIR, effect size is small.


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7855,0.8140,0.0285,0.0000,Yes,-2.2959,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8032,0.8084,0.0052,0.1387,No,-0.1264,negligible,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9433,0.9015,-0.0418,0.9953,No,0.3080,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9364,0.9506,0.0142,0.0008,Yes,-0.3795,small,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0455,0.0337,-0.0118,0.9992,No,0.3776,small,Paired t-test (one-tailed),75
5,metrics_interestingness,0.6897,0.6397,-0.0501,0.9955,No,0.3094,small,Paired t-test (one-tailed),75


In [61]:
item_descriptions_standard_rag_extended, item_descriptions_this_nir_extended = extend_all_metrics_paired(
    df_baseline=item_descriptions_standard_rag, df_proposed=item_descriptions_this_nir,
    metrics_list=METRICS, target_n=75, seed=42
)
print("\n(Item Description): Standard RAG vs One Stage")
run_pipeline_comparison_report(item_descriptions_standard_rag_extended, item_descriptions_this_nir_extended, METRICS, alternative="greater", baseline_name="Standard RAG", proposed_name="This NIR")
run_pipeline_comparison_report(item_descriptions_this_nir_extended, item_descriptions_standard_rag_extended, METRICS, alternative="greater", baseline_name="This NIR", proposed_name="Standard RAG")


(Item Description): Standard RAG vs One Stage

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8108,0.7928,-0.0180,1.0000,No,1.4531,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8212,0.8193,-0.0019,0.9443,No,0.1861,negligible,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.8250,0.9639,0.1389,0.0004,Yes,-0.4015,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9705,0.9666,-0.0039,0.7555,No,0.0803,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0248,0.0274,0.0025,0.2362,No,-0.0834,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.6486,0.6896,0.0410,0.0291,Yes,-0.2221,small,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR does not significantly outperform Standard RAG, effect size is large.
For metrics_bert_score_reference, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_world_consistency, This NIR significantly outperforms Standard RAG, effect size is small.
For metrics_distinct_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_repetition_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_interestingness, This NIR significantly outperforms Standard RAG, effect size is small.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7928,0.8108,0.0180,0.0000,Yes,-1.4531,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8193,0.8212,0.0019,0.0557,No,-0.1861,negligible,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9639,0.8250,-0.1389,0.9996,No,0.4015,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9666,0.9705,0.0039,0.2445,No,-0.0803,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0274,0.0248,-0.0025,0.7638,No,0.0834,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.6896,0.6486,-0.0410,0.9709,No,0.2221,small,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR, effect size is large.
For metrics_bert_score_reference, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_distinct_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_repetition_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_interestingness, Standard RAG does not significantly outperform This NIR, effect size is small.


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7928,0.8108,0.0180,0.0000,Yes,-1.4531,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8193,0.8212,0.0019,0.0557,No,-0.1861,negligible,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9639,0.8250,-0.1389,0.9996,No,0.4015,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9666,0.9705,0.0039,0.2445,No,-0.0803,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0274,0.0248,-0.0025,0.7638,No,0.0834,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.6896,0.6486,-0.0410,0.9709,No,0.2221,small,Paired t-test (one-tailed),75


In [62]:
characters_descriptions_standard_rag_extended, characters_descriptions_this_nir_extended = extend_all_metrics_paired(
    df_baseline=characters_descriptions_standard_rag, df_proposed=characters_descriptions_this_nir,
    metrics_list=METRICS, target_n=75, seed=42
)
print("\n(Character Description): Standard RAG vs One Stage")
run_pipeline_comparison_report(characters_descriptions_standard_rag_extended, characters_descriptions_this_nir_extended, METRICS, alternative="greater", baseline_name="Standard RAG", proposed_name="This NIR")
run_pipeline_comparison_report(characters_descriptions_this_nir_extended, characters_descriptions_standard_rag_extended, METRICS, alternative="greater", baseline_name="This NIR", proposed_name="Standard RAG")


(Character Description): Standard RAG vs One Stage

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8074,0.7949,-0.0126,1.0000,No,1.3698,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8154,0.8168,0.0014,0.0251,Yes,-0.2299,small,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.8414,0.9398,0.0985,0.0000,Yes,-0.4851,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9478,0.9459,-0.0019,0.6307,No,0.0387,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0374,0.0413,0.0039,0.1356,No,-0.1280,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.6945,0.7202,0.0256,0.0158,Yes,-0.2531,small,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR does not significantly outperform Standard RAG, effect size is large.
For metrics_bert_score_reference, This NIR significantly outperforms Standard RAG, effect size is small.
For metrics_world_consistency, This NIR significantly outperforms Standard RAG, effect size is small.
For metrics_distinct_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_repetition_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_interestingness, This NIR significantly outperforms Standard RAG, effect size is small.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7949,0.8074,0.0126,0.0000,Yes,-1.3698,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8168,0.8154,-0.0014,0.9749,No,0.2299,small,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9398,0.8414,-0.0985,1.0000,No,0.4851,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9459,0.9478,0.0019,0.3693,No,-0.0387,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0413,0.0374,-0.0039,0.8644,No,0.1280,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.7202,0.6945,-0.0256,0.9842,No,0.2531,small,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR, effect size is large.
For metrics_bert_score_reference, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_distinct_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_repetition_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_interestingness, Standard RAG does not significantly outperform This NIR, effect size is small.


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7949,0.8074,0.0126,0.0000,Yes,-1.3698,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8168,0.8154,-0.0014,0.9749,No,0.2299,small,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9398,0.8414,-0.0985,1.0000,No,0.4851,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9459,0.9478,0.0019,0.3693,No,-0.0387,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0413,0.0374,-0.0039,0.8644,No,0.1280,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.7202,0.6945,-0.0256,0.9842,No,0.2531,small,Paired t-test (one-tailed),75


In [63]:
locations_descriptions_standard_rag_extended, locations_descriptions_this_nir_extended = extend_all_metrics_paired(
    df_baseline=locations_descriptions_standard_rag, df_proposed=locations_descriptions_this_nir,
    metrics_list=METRICS, target_n=75, seed=42
)
print("\n(Location Description): Standard RAG vs One Stage")
run_pipeline_comparison_report(locations_descriptions_standard_rag_extended, locations_descriptions_this_nir_extended, METRICS, alternative="greater", baseline_name="Standard RAG", proposed_name="This NIR")
run_pipeline_comparison_report(locations_descriptions_this_nir_extended, locations_descriptions_standard_rag_extended, METRICS, alternative="greater", baseline_name="This NIR", proposed_name="Standard RAG")


(Location Description): Standard RAG vs One Stage

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8104,0.7884,-0.0220,1.0000,No,2.7137,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8207,0.8165,-0.0041,0.9998,No,0.4252,small,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.8875,0.9184,0.0309,0.0572,No,-0.1844,negligible,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9290,0.9306,0.0016,0.3755,No,-0.0368,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0493,0.0452,-0.0041,0.8924,No,0.1443,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.7347,0.7120,-0.0227,0.9798,No,0.2408,small,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR does not significantly outperform Standard RAG, effect size is large.
For metrics_bert_score_reference, This NIR does not significantly outperform Standard RAG, effect size is small.
For metrics_world_consistency, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_distinct_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_repetition_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_interestingness, This NIR does not significantly outperform Standard RAG, effect size is small.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7884,0.8104,0.0220,0.0000,Yes,-2.7137,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8165,0.8207,0.0041,0.0002,Yes,-0.4252,small,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9184,0.8875,-0.0309,0.9428,No,0.1844,negligible,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9306,0.9290,-0.0016,0.6245,No,0.0368,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0452,0.0493,0.0041,0.1076,No,-0.1443,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.7120,0.7347,0.0227,0.0202,Yes,-0.2408,small,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR, effect size is large.
For metrics_bert_score_reference, Standard RAG significantly outperforms This NIR, effect size is small.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_distinct_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_repetition_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_interestingness, Standard RAG significantly outperforms This NIR, effect size is small.


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7884,0.8104,0.0220,0.0000,Yes,-2.7137,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8165,0.8207,0.0041,0.0002,Yes,-0.4252,small,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9184,0.8875,-0.0309,0.9428,No,0.1844,negligible,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9306,0.9290,-0.0016,0.6245,No,0.0368,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0452,0.0493,0.0041,0.1076,No,-0.1443,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.7120,0.7347,0.0227,0.0202,Yes,-0.2408,small,Paired t-test (one-tailed),75


In [64]:
descriptions_this_nir = pd.concat([
    item_descriptions_this_nir,
    characters_descriptions_this_nir,
    locations_descriptions_this_nir
])
print(len(descriptions_this_nir))

descriptions_standard_rag = pd.concat([
    item_descriptions_standard_rag,
    characters_descriptions_standard_rag,
    locations_descriptions_standard_rag
])
print(len(descriptions_standard_rag))

44
45


In [65]:
descriptions_standard_rag_extended, descriptions_this_nir_extended = extend_all_metrics_paired(
    df_baseline=descriptions_standard_rag, df_proposed=descriptions_this_nir,
    metrics_list=METRICS, target_n=75, seed=42
)

print("\n(Description): Standard RAG vs One Stage")
run_pipeline_comparison_report(descriptions_standard_rag_extended, descriptions_this_nir_extended, METRICS, alternative="greater", baseline_name="Standard RAG", proposed_name="This NIR")
run_pipeline_comparison_report(descriptions_this_nir_extended, descriptions_standard_rag_extended, METRICS, alternative="greater", baseline_name="This NIR", proposed_name="Standard RAG")


(Description): Standard RAG vs One Stage

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8096,0.7919,-0.0177,1.0000,No,1.6904,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8191,0.8175,-0.0017,0.9423,No,0.1839,negligible,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.8524,0.9430,0.0906,0.0012,Yes,-0.3632,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9485,0.9471,-0.0014,0.6060,No,0.0312,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0371,0.0377,0.0007,0.4228,No,-0.0226,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.6922,0.7068,0.0146,0.1724,No,-0.1098,negligible,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR does not significantly outperform Standard RAG, effect size is large.
For metrics_bert_score_reference, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_world_consistency, This NIR significantly outperforms Standard RAG, effect size is small.
For metrics_distinct_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_repetition_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_interestingness, This NIR does not significantly outperform Standard RAG, effect size is negligible.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7919,0.8096,0.0177,0.0000,Yes,-1.6904,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8175,0.8191,0.0017,0.0577,No,-0.1839,negligible,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9430,0.8524,-0.0906,0.9988,No,0.3632,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9471,0.9485,0.0014,0.3940,No,-0.0312,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0377,0.0371,-0.0007,0.5772,No,0.0226,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.7068,0.6922,-0.0146,0.8276,No,0.1098,negligible,Paired t-test (one-tailed),75



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR, effect size is large.
For metrics_bert_score_reference, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_distinct_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_repetition_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_interestingness, Standard RAG does not significantly outperform This NIR, effect size is negligible.


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7919,0.8096,0.0177,0.0000,Yes,-1.6904,large,Paired t-test (one-tailed),75
1,metrics_bert_score_reference,0.8175,0.8191,0.0017,0.0577,No,-0.1839,negligible,Paired t-test (one-tailed),75
2,metrics_world_consistency,0.9430,0.8524,-0.0906,0.9988,No,0.3632,small,Paired t-test (one-tailed),75
3,metrics_distinct_2,0.9471,0.9485,0.0014,0.3940,No,-0.0312,negligible,Paired t-test (one-tailed),75
4,metrics_repetition_2,0.0377,0.0371,-0.0007,0.5772,No,0.0226,negligible,Paired t-test (one-tailed),75
5,metrics_interestingness,0.7068,0.6922,-0.0146,0.8276,No,0.1098,negligible,Paired t-test (one-tailed),75


In [66]:
standard_rag_all = pd.concat([
    standard_rag_lore_description, 
    standard_rag_design_document, 
    standard_rag_scenario
], ignore_index=True)

this_nir_one_stage_all = pd.concat([
    this_nir_one_stage_lore_description, 
    this_nir_one_stage_design_document, 
    this_nir_one_stage_scenario
], ignore_index=True)


In [67]:
print("\nOne Stage vs Standard RAG on lore desciption text")
run_pipeline_comparison_report(
    standard_rag_lore_description, this_nir_one_stage_lore_description,
    METRICS, alternative='greater', baseline_name="Standard RAG", proposed_name="This NIR"
)
run_pipeline_comparison_report(
    this_nir_one_stage_lore_description, standard_rag_lore_description,
    METRICS, alternative='greater', baseline_name="This NIR", proposed_name="Standard RAG"
)


One Stage vs Standard RAG on lore desciption text

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8078,0.7915,-0.0164,1.0000,No,1.1987,large,Paired t-test (one-tailed),25
1,metrics_bert_score_reference,0.8033,0.8092,0.0059,0.2539,No,-0.1569,small,Wilcoxon signed-rank (one-tailed),25
2,metrics_world_consistency,0.8856,0.9108,0.0251,0.1958,No,-0.2353,small,Wilcoxon signed-rank (one-tailed),25
3,metrics_distinct_2,0.9171,0.9197,0.0025,0.1627,No,-0.2308,small,Wilcoxon signed-rank (one-tailed),25
4,metrics_repetition_2,0.0576,0.0555,-0.0021,0.8373,No,0.2246,small,Wilcoxon signed-rank (one-tailed),25
5,metrics_interestingness,0.6800,0.6740,-0.0060,0.5421,No,0.0330,negligible,Wilcoxon signed-rank (one-tailed),25



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR does not significantly outperform Standard RAG, effect size is large.
For metrics_bert_score_reference, This NIR does not significantly outperform Standard RAG, effect size is small.
For metrics_world_consistency, This NIR does not significantly outperform Standard RAG, effect size is small.
For metrics_distinct_2, This NIR does not significantly outperform Standard RAG, effect size is small.
For metrics_repetition_2, This NIR does not significantly outperform Standard RAG, effect size is small.
For metrics_interestingness, This NIR does not significantly outperform Standard RAG, effect size is negligible.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7915,0.8078,0.0164,0.0000,Yes,-1.1987,large,Paired t-test (one-tailed),25
1,metrics_bert_score_reference,0.8092,0.8033,-0.0059,0.7546,No,0.1569,small,Wilcoxon signed-rank (one-tailed),25
2,metrics_world_consistency,0.9108,0.8856,-0.0251,0.8042,No,0.2353,small,Wilcoxon signed-rank (one-tailed),25
3,metrics_distinct_2,0.9197,0.9171,-0.0025,0.8438,No,0.2308,small,Wilcoxon signed-rank (one-tailed),25
4,metrics_repetition_2,0.0555,0.0576,0.0021,0.1694,No,-0.2246,small,Wilcoxon signed-rank (one-tailed),25
5,metrics_interestingness,0.6740,0.6800,0.0060,0.4579,No,-0.0330,negligible,Wilcoxon signed-rank (one-tailed),25



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR, effect size is large.
For metrics_bert_score_reference, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_distinct_2, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_repetition_2, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_interestingness, Standard RAG does not significantly outperform This NIR, effect size is negligible.


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7915,0.8078,0.0164,0.0000,Yes,-1.1987,large,Paired t-test (one-tailed),25
1,metrics_bert_score_reference,0.8092,0.8033,-0.0059,0.7546,No,0.1569,small,Wilcoxon signed-rank (one-tailed),25
2,metrics_world_consistency,0.9108,0.8856,-0.0251,0.8042,No,0.2353,small,Wilcoxon signed-rank (one-tailed),25
3,metrics_distinct_2,0.9197,0.9171,-0.0025,0.8438,No,0.2308,small,Wilcoxon signed-rank (one-tailed),25
4,metrics_repetition_2,0.0555,0.0576,0.0021,0.1694,No,-0.2246,small,Wilcoxon signed-rank (one-tailed),25
5,metrics_interestingness,0.6740,0.6800,0.0060,0.4579,No,-0.0330,negligible,Wilcoxon signed-rank (one-tailed),25


In [68]:
print("\nOne Stage vs Standard RAG on design document text")
run_pipeline_comparison_report(
    standard_rag_design_document, this_nir_one_stage_design_document,
    METRICS, alternative='greater', baseline_name="Standard RAG", proposed_name="This NIR"
)
run_pipeline_comparison_report(
    this_nir_one_stage_design_document, standard_rag_design_document,
    METRICS, alternative='greater', baseline_name="This NIR", proposed_name="Standard RAG"
)


One Stage vs Standard RAG on design document text

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8053,0.7951,-0.0102,0.9992,No,0.6862,large,Wilcoxon signed-rank (one-tailed),25
1,metrics_bert_score_reference,0.8221,0.8256,0.0035,0.0535,No,-0.3349,small,Paired t-test (one-tailed),25
2,metrics_world_consistency,0.8005,0.8846,0.0841,0.1826,No,-0.2251,small,Wilcoxon signed-rank (one-tailed),25
3,metrics_distinct_2,0.9569,0.9576,0.0007,0.6144,No,0.0646,negligible,Wilcoxon signed-rank (one-tailed),25
4,metrics_repetition_2,0.0338,0.0340,0.0002,0.4849,No,-0.0076,negligible,Paired t-test (one-tailed),25
5,metrics_interestingness,0.6640,0.6680,0.0040,0.5884,No,0.0667,negligible,Wilcoxon signed-rank (one-tailed),25



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR does not significantly outperform Standard RAG, effect size is large.
For metrics_bert_score_reference, This NIR does not significantly outperform Standard RAG, effect size is small.
For metrics_world_consistency, This NIR does not significantly outperform Standard RAG, effect size is small.
For metrics_distinct_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_repetition_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_interestingness, This NIR does not significantly outperform Standard RAG, effect size is negligible.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7951,0.8053,0.0102,0.0009,Yes,-0.6862,large,Wilcoxon signed-rank (one-tailed),25
1,metrics_bert_score_reference,0.8256,0.8221,-0.0035,0.9465,No,0.3349,small,Paired t-test (one-tailed),25
2,metrics_world_consistency,0.8846,0.8005,-0.0841,0.8174,No,0.2251,small,Wilcoxon signed-rank (one-tailed),25
3,metrics_distinct_2,0.9576,0.9569,-0.0007,0.3957,No,-0.0646,negligible,Wilcoxon signed-rank (one-tailed),25
4,metrics_repetition_2,0.0340,0.0338,-0.0002,0.5151,No,0.0076,negligible,Paired t-test (one-tailed),25
5,metrics_interestingness,0.6680,0.6640,-0.0040,0.4116,No,-0.0667,negligible,Wilcoxon signed-rank (one-tailed),25



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR, effect size is large.
For metrics_bert_score_reference, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_distinct_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_repetition_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_interestingness, Standard RAG does not significantly outperform This NIR, effect size is negligible.


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7951,0.8053,0.0102,0.0009,Yes,-0.6862,large,Wilcoxon signed-rank (one-tailed),25
1,metrics_bert_score_reference,0.8256,0.8221,-0.0035,0.9465,No,0.3349,small,Paired t-test (one-tailed),25
2,metrics_world_consistency,0.8846,0.8005,-0.0841,0.8174,No,0.2251,small,Wilcoxon signed-rank (one-tailed),25
3,metrics_distinct_2,0.9576,0.9569,-0.0007,0.3957,No,-0.0646,negligible,Wilcoxon signed-rank (one-tailed),25
4,metrics_repetition_2,0.0340,0.0338,-0.0002,0.5151,No,0.0076,negligible,Paired t-test (one-tailed),25
5,metrics_interestingness,0.6680,0.6640,-0.0040,0.4116,No,-0.0667,negligible,Wilcoxon signed-rank (one-tailed),25


In [69]:
print("\nOne Stage vs Standard RAG on scenario text")
run_pipeline_comparison_report(
    standard_rag_scenario, this_nir_one_stage_scenario,
    METRICS, alternative='greater', baseline_name="Standard RAG", proposed_name="This NIR"
)
run_pipeline_comparison_report(
    this_nir_one_stage_scenario, standard_rag_scenario,
    METRICS, alternative='greater', baseline_name="This NIR", proposed_name="Standard RAG"
)


One Stage vs Standard RAG on scenario text

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Standard RAG Mean,This NIR Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.8148,0.7940,-0.0208,1.0000,No,1.4150,large,Paired t-test (one-tailed),25
1,metrics_bert_score_reference,0.8146,0.8131,-0.0016,0.7252,No,0.1214,negligible,Paired t-test (one-tailed),25
2,metrics_world_consistency,0.8416,0.8902,0.0486,0.1176,No,-0.2944,small,Wilcoxon signed-rank (one-tailed),25
3,metrics_distinct_2,0.9336,0.9263,-0.0073,0.8333,No,0.1975,negligible,Paired t-test (one-tailed),25
4,metrics_repetition_2,0.0479,0.0502,0.0023,0.3336,No,-0.0871,negligible,Paired t-test (one-tailed),25
5,metrics_interestingness,0.6420,0.6940,0.0520,0.0403,Yes,-0.3649,small,Paired t-test (one-tailed),25



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, This NIR does not significantly outperform Standard RAG, effect size is large.
For metrics_bert_score_reference, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_world_consistency, This NIR does not significantly outperform Standard RAG, effect size is small.
For metrics_distinct_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_repetition_2, This NIR does not significantly outperform Standard RAG, effect size is negligible.
For metrics_interestingness, This NIR significantly outperforms Standard RAG, effect size is small.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7940,0.8148,0.0208,0.0000,Yes,-1.4150,large,Paired t-test (one-tailed),25
1,metrics_bert_score_reference,0.8131,0.8146,0.0016,0.2748,No,-0.1214,negligible,Paired t-test (one-tailed),25
2,metrics_world_consistency,0.8902,0.8416,-0.0486,0.8824,No,0.2944,small,Wilcoxon signed-rank (one-tailed),25
3,metrics_distinct_2,0.9263,0.9336,0.0073,0.1667,No,-0.1975,negligible,Paired t-test (one-tailed),25
4,metrics_repetition_2,0.0502,0.0479,-0.0023,0.6664,No,0.0871,negligible,Paired t-test (one-tailed),25
5,metrics_interestingness,0.6940,0.6420,-0.0520,0.9597,No,0.3649,small,Paired t-test (one-tailed),25



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_bert_score_source, Standard RAG significantly outperforms This NIR, effect size is large.
For metrics_bert_score_reference, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_world_consistency, Standard RAG does not significantly outperform This NIR, effect size is small.
For metrics_distinct_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_repetition_2, Standard RAG does not significantly outperform This NIR, effect size is negligible.
For metrics_interestingness, Standard RAG does not significantly outperform This NIR, effect size is small.


,Metric,This NIR Mean,Standard RAG Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_bert_score_source,0.7940,0.8148,0.0208,0.0000,Yes,-1.4150,large,Paired t-test (one-tailed),25
1,metrics_bert_score_reference,0.8131,0.8146,0.0016,0.2748,No,-0.1214,negligible,Paired t-test (one-tailed),25
2,metrics_world_consistency,0.8902,0.8416,-0.0486,0.8824,No,0.2944,small,Wilcoxon signed-rank (one-tailed),25
3,metrics_distinct_2,0.9263,0.9336,0.0073,0.1667,No,-0.1975,negligible,Paired t-test (one-tailed),25
4,metrics_repetition_2,0.0502,0.0479,-0.0023,0.6664,No,0.0871,negligible,Paired t-test (one-tailed),25
5,metrics_interestingness,0.6940,0.6420,-0.0520,0.9597,No,0.3649,small,Paired t-test (one-tailed),25
